# ⚽ WSL Shot Analytics — Five-Model xG System

**Models**
| # | Model | What it answers |
|---|---|---|
| 1 | Pre-Shot xG | How likely is this shot to score? |
| 2 | Post-Shot psxG | Given where it was aimed, how likely? |
| 3 | Situation Danger | How dangerous was the build-up? |
| 4 | P(On Target) | Will this shot hit the frame? |
| 5 | xGOT | Given it's on target, will it score? |
| 6 | Multi-Outcome | P(goal \| save \| miss \| post) |

**Shot metrics per row:** placement_score · shot_power · contact_quality · optimal_placement · technique_index · save_difficulty · finishing_luck · rolling form · archetype

---
**Instructions**
1. Upload your WSL JSON folder to Google Drive
2. Run **Section 1** to install packages and mount Drive
3. Set `DATA_ROOT` in **Section 2** to your Drive folder
4. Run all cells top-to-bottom (`Runtime → Run all`)

## 1 · Setup

In [ ]:
# Install required packages (only needed once per Colab session)
!pip install xgboost lightgbm shap mplsoccer scikit-learn joblib -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, glob, json, warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, GridSearchCV
from sklearn.calibration import calibration_curve
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import roc_curve, auc, brier_score_loss, roc_auc_score, log_loss
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import joblib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from mplsoccer import Pitch, VerticalPitch

import xgboost as xgb

try:
    import shap
    HAS_SHAP = True
except ImportError:
    HAS_SHAP = False
    print('shap not installed — SHAP plots skipped')

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

print('All packages loaded ✓')

## 2 · Configuration
**Set `DATA_ROOT` to the folder containing your WSL season subfolders.**

In [ ]:
# ── CONFIGURE THESE PATHS ────────────────────────────────────────────────────
DATA_ROOT   = '/content/drive/MyDrive/Project-Beth-Mead'   # folder with WSL 20XX subfolders
OUTPUT_DIR  = '/content/drive/MyDrive/Project-Beth-Mead/xg_output'
PLAYER_NAME = 'Mead'   # substring to filter for player deep-dive
# ─────────────────────────────────────────────────────────────────────────────

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Visual palette
FIG_BG   = '#0e1117'
PITCH_BG = '#0d1117'
LINE_COL = '#c9d1d9'
C_BLUE   = '#58a6ff'
C_ORANGE = '#f78166'
C_GREEN  = '#3fb950'
C_YELLOW = '#e3b341'
C_PURPLE = '#bc8cff'
C_MUTED  = '#484f58'

# Pitch / goal constants (Opta coordinates)
GOAL_X        = 100.0
GOAL_Y_LEFT   = 44.0
GOAL_Y_RIGHT  = 56.0
GOAL_Y_CENTRE = 50.0
GOAL_WIDTH    = 12.0
GOAL_H_HIGH   = 62.0

# Qualifier IDs (verified against this dataset)
Q = dict(
    PENALTY=9, HEADER=15, RIGHT_FOOT=20, LEFT_FOOT=72,
    REGULAR_PLAY=22, FAST_BREAK=23, SET_PIECE=24, FROM_CORNER=25,
    FREE_KICK=26, DIRECT_FK=28, VOLLEY=108, DEFLECTION=133,
    PULL_BACK=195, BIG_CHANCE=233, FIRST_TIME=200,
    GOAL_Y=102, GOAL_HEIGHT=231, SAVE_END_X=146, SAVE_END_Y=147,
    UNDER_PRESSURE=18, INTENTIONAL=154, BODY_SIDE=56,
)

print(f'Output dir: {OUTPUT_DIR}')

## 3 · Geometry & Loading Helpers

In [ ]:
def open_angle(x, y):
    shot  = np.array([x, y], dtype=float)
    left  = np.array([GOAL_X, GOAL_Y_LEFT],  dtype=float)
    right = np.array([GOAL_X, GOAL_Y_RIGHT], dtype=float)
    v1, v2 = left - shot, right - shot
    n1, n2 = np.linalg.norm(v1), np.linalg.norm(v2)
    if n1 == 0 or n2 == 0: return 0.0
    return float(np.degrees(np.arccos(np.clip(np.dot(v1,v2)/(n1*n2),-1,1))))

def dist_to_goal(x, y):
    return float(np.sqrt((GOAL_X-x)**2 + (y-GOAL_Y_CENTRE)**2))

def placement_score(gy_norm, gh_norm):
    if pd.isna(gy_norm) or pd.isna(gh_norm): return np.nan
    gy = float(np.clip(gy_norm, -1, 1))
    gh = float(np.clip(gh_norm,  0, 1))
    keeper_h = 0.40
    lat  = abs(gy)
    vert = abs(gh - keeper_h) / max(keeper_h, 1 - keeper_h)
    return float(np.clip(np.sqrt(0.6*lat**2 + 0.4*vert**2)*100, 0, 100))

def goal_zone(goal_y, goal_h):
    if pd.isna(goal_y) or pd.isna(goal_h): return 'Unknown'
    h = 'Top' if goal_h >= GOAL_H_HIGH else 'Bottom'
    s = 'Left' if goal_y < 47.5 else ('Right' if goal_y > 52.5 else 'Centre')
    return f'{h} {s}'

def get_q(event, qid):
    for q in event.get('qualifier', []):
        if q['qualifierId'] == qid: return q.get('value', 1)
    return None

def has_q(event, qid):
    return any(q['qualifierId'] == qid for q in event.get('qualifier', []))

def safe_float(v):
    try: return float(v)
    except: return np.nan

def season_from_path(path):
    for p in path.replace('\\','/').split('/'):
        if p.startswith('WSL'): return p
    return 'Unknown'

print('Helpers defined ✓')

In [ ]:
def load_match(path):
    with open(path, 'r', encoding='utf-8-sig') as f:
        data = json.load(f)
    events = data.get('liveData', {}).get('event', data.get('event', []))
    rows = []
    for e in events:
        tid = str(e.get('typeId'))
        if tid not in ('13','14','15','16'): continue
        x = safe_float(e.get('x')); y = safe_float(e.get('y'))
        if np.isnan(x) or np.isnan(y): continue

        gy_raw  = safe_float(get_q(e, Q['GOAL_Y']))
        gh_raw  = safe_float(get_q(e, Q['GOAL_HEIGHT']))
        gy_norm = (gy_raw - GOAL_Y_CENTRE)/(GOAL_WIDTH/2) if not np.isnan(gy_raw) else np.nan
        gh_norm = gh_raw/100.0 if not np.isnan(gh_raw) else np.nan

        if not np.isnan(gy_norm) and not np.isnan(gh_norm):
            gf_dist = np.sqrt(gy_norm**2 + (gh_norm-0.35)**2)
            corner  = int(abs(gy_norm)>0.55 or gh_norm>0.60)
            ps      = placement_score(gy_norm, gh_norm)
        elif not np.isnan(gy_norm):
            gf_dist = abs(gy_norm); corner = int(abs(gy_norm)>0.55); ps = abs(gy_norm)*100
        else:
            gf_dist = corner = ps = np.nan

        body_side = str(get_q(e, Q['BODY_SIDE']) or '').strip()
        is_rf = has_q(e, Q['RIGHT_FOOT']); is_lf = has_q(e, Q['LEFT_FOOT'])
        weak_foot = int((is_rf and body_side=='Left') or (is_lf and body_side=='Right'))

        dist  = dist_to_goal(x, y)
        angle = open_angle(x, y)
        y_sym = abs(y - GOAL_Y_CENTRE)

        rows.append({
            'match_file': os.path.basename(path),
            'season': season_from_path(path),
            'player_id': str(e.get('playerId','')),
            'player_name': e.get('playerName',''),
            'contestant_id': str(e.get('contestantId','')),
            'type_id': int(tid),
            'period_id': int(e.get('periodId') or 0),
            'time_min': int(e.get('timeMin') or 0),
            'x': x, 'y': y, 'y_sym': y_sym,
            'distance': dist, 'log_distance': np.log(max(dist,0.5)),
            'angle': angle, 'angle_sin': np.sin(np.radians(angle)),
            'in_six_yard': int(x>=94.2 and 36.8<=y<=63.2),
            'in_penalty_box': int(x>=83.0 and 21.1<=y<=78.9),
            'central_y': int(y_sym < GOAL_WIDTH/2),
            'dist_to_post': abs(y_sym - GOAL_WIDTH/2),
            'goal_y_raw': gy_raw, 'goal_y_norm': gy_norm,
            'goal_h_raw': gh_raw, 'goal_h_norm': gh_norm,
            'goal_frame_dist': gf_dist, 'corner_zone': corner,
            'placement_score': ps, 'goal_zone': goal_zone(gy_raw, gh_raw),
            'is_header': int(has_q(e,Q['HEADER'])),
            'is_right_foot': int(is_rf), 'is_left_foot': int(is_lf),
            'weak_foot': weak_foot,
            'is_volley': int(has_q(e,Q['VOLLEY'])),
            'is_deflected': int(has_q(e,Q['DEFLECTION'])),
            'is_first_time': int(has_q(e,Q['FIRST_TIME'])),
            'is_big_chance': int(has_q(e,Q['BIG_CHANCE'])),
            'is_fast_break': int(has_q(e,Q['FAST_BREAK'])),
            'is_from_corner': int(has_q(e,Q['FROM_CORNER'])),
            'is_free_kick': int(has_q(e,Q['DIRECT_FK'])),
            'is_penalty': int(has_q(e,Q['PENALTY'])),
            'is_set_piece': int(has_q(e,Q['SET_PIECE'])),
            'is_open_play': int(has_q(e,Q['REGULAR_PLAY'])),
            'is_pull_back': int(has_q(e,Q['PULL_BACK'])),
            'under_pressure': int(has_q(e,Q['UNDER_PRESSURE'])),
            'is_intentional': int(has_q(e,Q['INTENTIONAL'])),
            'is_goal': int(tid=='16'),
            'is_on_target': int(tid in ('15','16')),
            'is_post': int(tid=='14'),
            'is_blocked': int(tid=='15'),
        })
    return pd.DataFrame(rows)

print('load_match defined ✓')

## 4 · Load All Matches

In [ ]:
json_files = glob.glob(os.path.join(DATA_ROOT, '**', '*.json'), recursive=True)
print(f'Found {len(json_files)} JSON files')

frames = []
errors = []
for path in json_files:
    try:
        frames.append(load_match(path))
    except Exception as exc:
        errors.append((os.path.basename(path), str(exc)))

if errors:
    print(f'Skipped {len(errors)} files with errors')

shots = pd.concat(frames, ignore_index=True)

# Technique index
def technique_index(row):
    s = 0
    if row['is_header']:      s += 20
    if row['is_volley']:      s += 15
    if row['is_first_time']:  s += 10
    if row['under_pressure']: s += 25
    if row['is_deflected']:   s += 10
    if row['weak_foot']:      s += 20
    return float(min(s, 100))

shots['technique_index'] = shots.apply(technique_index, axis=1)

print(f"Loaded {len(shots):,} shots | Goals: {shots['is_goal'].sum():,} ({shots['is_goal'].mean()*100:.1f}%)")
print(f"Seasons: {sorted(shots['season'].unique())}")
shots.head(3)

## 5 · Model Infrastructure

In [ ]:
class CalibratedXGB:
    """XGBoost + held-out isotonic calibration."""
    def __init__(self, clf, iso):
        self.clf = clf; self.iso = iso
    def predict_proba(self, X):
        raw = self.clf.predict_proba(X)[:, 1]
        cal = self.iso.predict(raw)
        return np.column_stack([1-cal, cal])

def train_model(X_tr, y_tr, g_tr, X_cal, y_cal, X_test, y_test, label):
    group_cv = GroupKFold(n_splits=5)
    spw  = (y_tr==0).sum() / max((y_tr==1).sum(), 1)
    grid = GridSearchCV(
        xgb.XGBClassifier(
            objective='binary:logistic', eval_metric='auc',
            use_label_encoder=False, random_state=42, verbosity=0,
        ),
        {
            'max_depth': [4, 6], 'learning_rate': [0.05, 0.1],
            'n_estimators': [300, 500], 'subsample': [0.8],
            'colsample_bytree': [0.7, 0.9], 'min_child_weight': [5, 10],
            'scale_pos_weight': [spw],
        },
        scoring='roc_auc', cv=group_cv, n_jobs=-1, verbose=0,
    )
    grid.fit(X_tr, y_tr, groups=g_tr)
    clf = grid.best_estimator_
    iso = IsotonicRegression(out_of_bounds='clip')
    iso.fit(clf.predict_proba(X_cal)[:, 1], y_cal)
    model = CalibratedXGB(clf, iso)
    p_test = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, p_test)
    m_auc   = auc(fpr, tpr)
    m_brier = brier_score_loss(y_test, p_test)
    print(f'  [{label}]  CV AUC: {grid.best_score_:.4f}  |  Test AUC: {m_auc:.4f}  Brier: {m_brier:.4f}')
    return model, p_test, fpr, tpr, m_auc, m_brier

# Penalties — empirical mean only
pen_mask = shots['is_penalty'] == 1
nonpen   = shots[~pen_mask].copy().reset_index(drop=True)
pen_xg   = shots.loc[pen_mask, 'is_goal'].mean() if pen_mask.sum() > 0 else 0.76
for col in ['xg','psxg','situation_danger','p_ontarget','xgot']:
    shots.loc[pen_mask, col] = pen_xg
print(f'Penalty xG (empirical): {pen_xg:.3f}  n={pen_mask.sum()}')

# Feature sets
XG_FEATURES = [
    'x','y_sym','distance','log_distance','angle','angle_sin',
    'dist_to_post','in_six_yard','in_penalty_box','central_y',
    'is_header','is_right_foot','is_left_foot','weak_foot',
    'is_volley','is_deflected','is_first_time','is_big_chance',
    'is_fast_break','is_from_corner','is_free_kick','is_set_piece',
    'is_open_play','is_pull_back','under_pressure','is_intentional',
]
PSXG_FEATURES = XG_FEATURES + ['goal_y_norm','goal_h_norm','goal_frame_dist','corner_zone','placement_score']
SIT_FEATURES  = ['is_big_chance','is_fast_break','is_from_corner','is_free_kick',
                 'is_set_piece','is_open_play','is_pull_back','is_first_time',
                 'is_header','under_pressure','is_intentional','period_id','time_min']
POT_FEATURES  = ['x','y_sym','distance','log_distance','angle','angle_sin',
                 'dist_to_post','in_six_yard','in_penalty_box','central_y',
                 'is_header','is_right_foot','is_left_foot','weak_foot',
                 'is_volley','is_deflected','is_first_time','is_big_chance',
                 'is_fast_break','is_from_corner','is_open_play','under_pressure']

X_base = nonpen[XG_FEATURES].fillna(0).astype(float)
X_sit  = nonpen[SIT_FEATURES].fillna(0).astype(float)
X_pot  = nonpen[POT_FEATURES].fillna(0).astype(float)
y_goal = nonpen['is_goal'].astype(int)
y_ot   = nonpen['is_on_target'].astype(int)
groups = nonpen['match_file']

gss_outer = GroupShuffleSplit(1, test_size=0.20, random_state=42)
dev_idx, test_idx = next(gss_outer.split(X_base, y_goal, groups=groups))
gss_inner = GroupShuffleSplit(1, test_size=0.25, random_state=0)
tr_idx, cal_idx = next(gss_inner.split(
    X_base.iloc[dev_idx], y_goal.iloc[dev_idx], groups=groups.iloc[dev_idx]
))

def split(X):
    Xd = X.iloc[dev_idx]
    return Xd.iloc[tr_idx], Xd.iloc[cal_idx], X.iloc[test_idx]

X_base_tr, X_base_cal, X_base_test = split(X_base)
X_sit_tr,  X_sit_cal,  X_sit_test  = split(X_sit)
X_pot_tr,  X_pot_cal,  X_pot_test  = split(X_pot)
y_tr   = y_goal.iloc[dev_idx].iloc[tr_idx]
y_cal  = y_goal.iloc[dev_idx].iloc[cal_idx]
y_test = y_goal.iloc[test_idx]
y_ot_tr   = y_ot.iloc[dev_idx].iloc[tr_idx]
y_ot_cal  = y_ot.iloc[dev_idx].iloc[cal_idx]
y_ot_test = y_ot.iloc[test_idx]
g_tr   = groups.iloc[dev_idx].iloc[tr_idx]

print(f'Train: {len(y_tr):,}  Cal: {len(y_cal):,}  Test: {len(y_test):,}')

## 6 · Train Five Models
> ⏱ This cell takes ~5–10 minutes (grid search over 5-fold GroupKFold × 32 param combos)

In [ ]:
print('── Model 1: Pre-Shot xG ──')
xg_model, xg_p, xg_fpr, xg_tpr, xg_auc, xg_brier = train_model(
    X_base_tr, y_tr, g_tr, X_base_cal, y_cal, X_base_test, y_test, 'xG')
nonpen['xg'] = xg_model.predict_proba(X_base)[:, 1]
shots.loc[~pen_mask, 'xg'] = nonpen['xg'].values
joblib.dump(xg_model, os.path.join(OUTPUT_DIR, 'model_xg.pkl'))

print('\n── Model 2: Post-Shot psxG ──')
has_pl  = nonpen[['goal_y_norm','goal_h_norm']].notna().any(axis=1)
nps     = nonpen[has_pl].reset_index(drop=True)
X_ps    = nps[PSXG_FEATURES].fillna(0).astype(float)
y_ps    = nps['is_goal'].astype(int); g_ps = nps['match_file']
print(f'  Shots with placement: {len(nps):,}/{len(nonpen):,}')
g1 = GroupShuffleSplit(1,test_size=0.20,random_state=42)
dp,tp = next(g1.split(X_ps,y_ps,groups=g_ps))
g2 = GroupShuffleSplit(1,test_size=0.25,random_state=0)
trp,cap = next(g2.split(X_ps.iloc[dp],y_ps.iloc[dp],groups=g_ps.iloc[dp]))
psxg_model,psxg_p,psxg_fpr,psxg_tpr,psxg_auc,psxg_brier = train_model(
    X_ps.iloc[dp].iloc[trp],y_ps.iloc[dp].iloc[trp],g_ps.iloc[dp].iloc[trp],
    X_ps.iloc[dp].iloc[cap],y_ps.iloc[dp].iloc[cap],X_ps.iloc[tp],y_ps.iloc[tp],'psxG')
X_all_ps = nonpen[PSXG_FEATURES].fillna(0).astype(float)
nonpen['psxg'] = nonpen['xg'].copy()
nonpen.loc[has_pl,'psxg'] = psxg_model.predict_proba(X_all_ps[has_pl])[:,1]
shots.loc[~pen_mask,'psxg'] = nonpen['psxg'].values
joblib.dump(psxg_model, os.path.join(OUTPUT_DIR,'model_psxg.pkl'))

print('\n── Model 3: Situation Danger ──')
sit_model,sit_p,sit_fpr,sit_tpr,sit_auc,sit_brier = train_model(
    X_sit_tr,y_tr,g_tr,X_sit_cal,y_cal,X_sit_test,y_test,'Situation')
nonpen['situation_danger'] = sit_model.predict_proba(X_sit)[:,1]
shots.loc[~pen_mask,'situation_danger'] = nonpen['situation_danger'].values
joblib.dump(sit_model, os.path.join(OUTPUT_DIR,'model_situation.pkl'))

print('\n── Model 4: P(On Target) ──')
pot_model,pot_p,pot_fpr,pot_tpr,pot_auc,pot_brier = train_model(
    X_pot_tr,y_ot_tr,g_tr,X_pot_cal,y_ot_cal,X_pot_test,y_ot_test,'P(OT)')
nonpen['p_ontarget'] = pot_model.predict_proba(X_pot)[:,1]
shots.loc[~pen_mask,'p_ontarget'] = nonpen['p_ontarget'].values
joblib.dump(pot_model, os.path.join(OUTPUT_DIR,'model_pontarget.pkl'))

print('\n── Model 5: xGOT ──')
ot_mask = nonpen['is_on_target']==1
not_    = nonpen[ot_mask].reset_index(drop=True)
X_ot    = not_[PSXG_FEATURES].fillna(0).astype(float)
y_otg   = not_['is_goal'].astype(int); g_ot = not_['match_file']
print(f'  On-target: {len(not_):,}  Goals: {y_otg.sum():,} ({y_otg.mean()*100:.1f}%)')
go = GroupShuffleSplit(1,test_size=0.20,random_state=42)
do,to = next(go.split(X_ot,y_otg,groups=g_ot))
gi = GroupShuffleSplit(1,test_size=0.25,random_state=0)
tri,cai = next(gi.split(X_ot.iloc[do],y_otg.iloc[do],groups=g_ot.iloc[do]))
xgot_model,xgot_p,xgot_fpr,xgot_tpr,xgot_auc,xgot_brier = train_model(
    X_ot.iloc[do].iloc[tri],y_otg.iloc[do].iloc[tri],g_ot.iloc[do].iloc[tri],
    X_ot.iloc[do].iloc[cai],y_otg.iloc[do].iloc[cai],X_ot.iloc[to],y_otg.iloc[to],'xGOT')
nonpen['xgot'] = 0.0
nonpen.loc[ot_mask,'xgot'] = xgot_model.predict_proba(X_ot)[:,1]
shots.loc[~pen_mask,'xgot'] = nonpen['xgot'].values
shots.loc[pen_mask,'xgot']  = pen_xg
joblib.dump(xgot_model, os.path.join(OUTPUT_DIR,'model_xgot.pkl'))

print('\n✓ All five models trained and saved')

## 7 · Derived Shot Metrics

In [ ]:
shots['placement_quality'] = shots['psxg'] - shots['xg']
shots['execution_quality'] = shots['xg']  - shots['situation_danger']
shots['ot_over_exp']       = shots['is_on_target'] - shots.get('p_ontarget', 0)
shots['save_difficulty']   = np.where(shots['is_blocked']==1, shots['psxg'], np.nan)
shots['finishing_luck']    = shots['is_goal'] - shots['psxg']

def shot_power(row):
    p = 50.0
    if row['is_volley']:      p += 25
    if row['is_first_time']:  p += 12
    if row['is_header']:      p -= 8
    if row['weak_foot']:      p -= 12
    if row['under_pressure']: p -= 8
    if row['is_deflected']:   p += 5
    return float(np.clip(p, 0, 100))

def contact_quality(psxg, tech, ps):
    if pd.isna(psxg) or pd.isna(ps): return float(psxg*100) if not pd.isna(psxg) else np.nan
    return float(np.clip((0.50*(ps/100) + 0.30*psxg + 0.20*(tech/60))*100, 0, 100))

shots['shot_power']      = shots.apply(shot_power, axis=1)
shots['contact_quality'] = shots.apply(
    lambda r: contact_quality(r['psxg'], r['technique_index'], r['placement_score']), axis=1)

for col in ['xg','psxg','situation_danger','p_ontarget','xgot']:
    if col in shots.columns:
        mn, mx = shots[col].min(), shots[col].max()
        shots[f'{col}_index'] = (shots[col]-mn)/max(mx-mn,1e-9)*100

print('Derived metrics added ✓')
shots[['player_name','season','xg','psxg','xgot','placement_score',
       'shot_power','contact_quality','goal_zone']].tail(5)

## 8 · Multi-Outcome Model (goal | save | miss | post)

In [ ]:
MO_FEATURES = XG_FEATURES.copy()
OUTCOME_MAP  = {13:0, 14:1, 15:2, 16:3}
OUTCOME_NAME = {0:'Miss', 1:'Post', 2:'Save', 3:'Goal'}

X_mo = nonpen[MO_FEATURES].fillna(0).astype(float)
y_mo = nonpen['type_id'].map(OUTCOME_MAP).astype(int)
g_mo = nonpen['match_file']

gm = GroupShuffleSplit(1,test_size=0.20,random_state=42)
dm,tm = next(gm.split(X_mo,y_mo,groups=g_mo))
gm2 = GroupShuffleSplit(1,test_size=0.25,random_state=0)
trm,cam = next(gm2.split(X_mo.iloc[dm],y_mo.iloc[dm],groups=g_mo.iloc[dm]))

mo_clf = xgb.XGBClassifier(
    objective='multi:softprob', num_class=4, eval_metric='mlogloss',
    use_label_encoder=False, max_depth=5, learning_rate=0.05,
    n_estimators=400, subsample=0.8, colsample_bytree=0.8,
    min_child_weight=5, random_state=42, verbosity=0, n_jobs=-1,
)
mo_clf.fit(X_mo.iloc[dm].iloc[trm], y_mo.iloc[dm].iloc[trm])
test_proba = mo_clf.predict_proba(X_mo.iloc[tm])
print(f'Log-loss: {log_loss(y_mo.iloc[tm], test_proba):.4f}')
for c, name in OUTCOME_NAME.items():
    yb = (y_mo.iloc[tm]==c).astype(int)
    if yb.sum() > 0:
        print(f'  {name}: AUC={roc_auc_score(yb, test_proba[:,c]):.4f}')

all_proba = mo_clf.predict_proba(X_mo)
for i, col in enumerate(['p_miss','p_post','p_save','p_goal_mo']):
    shots.loc[~pen_mask, col] = all_proba[:, i]
joblib.dump(mo_clf, os.path.join(OUTPUT_DIR,'model_multi_outcome.pkl'))
print('Multi-outcome model saved ✓')

## 9 · Shot Archetypes & Rolling Form

In [ ]:
# ── Shot archetypes (k-means, k=6) ──────────────────────────────────────────
CLUSTER_FEATURES = ['x','y_sym','log_distance','angle_sin',
                    'is_header','is_first_time','is_volley','under_pressure',
                    'goal_y_norm','goal_h_norm']
cd = shots[CLUSTER_FEATURES].copy()
cd['goal_y_norm'] = cd['goal_y_norm'].clip(-1,1).fillna(0.0)
cd['goal_h_norm'] = cd['goal_h_norm'].clip(0,1).fillna(0.35)

scaler = StandardScaler()
X_cl   = scaler.fit_transform(cd.fillna(0).values)
km     = KMeans(n_clusters=6, random_state=42, n_init=20)
shots['archetype_id'] = km.fit_predict(X_cl)

centres = scaler.inverse_transform(km.cluster_centers_)
cdf     = pd.DataFrame(centres, columns=CLUSTER_FEATURES)
def name_arch(row):
    if row['is_header'] > 0.4:          return 'Aerial Header'
    if row['x'] >= 88 and row['y_sym'] < 6: return 'Close-Range Central'
    if row['log_distance'] > np.log(22): return 'Long-Range Effort'
    if row['under_pressure'] > 0.5:     return 'Pressured Attempt'
    if row['is_first_time'] > 0.25 or row['is_volley'] > 0.25: return 'First-Time / Volley'
    return 'Placed Box Finish'
archetype_names = {i: name_arch(cdf.iloc[i]) for i in range(6)}
seen = {}
for k, v in archetype_names.items():
    n = seen.get(v, 0)
    archetype_names[k] = f'{v} ({n+1})' if n > 0 else v
    seen[v] = n + 1
shots['archetype'] = shots['archetype_id'].map(archetype_names)

print('Archetypes:')
print(shots.groupby('archetype')[['is_goal','xg','psxg','shot_power']]
        .agg({'is_goal':['count','mean'],'xg':'mean','psxg':'mean','shot_power':'mean'})
        .round(3))

# ── Rolling form ─────────────────────────────────────────────────────────────
shots_s = shots.sort_values(['player_name','season','time_min']).copy()
for col, src in [('rolling_xg10','xg'),('rolling_psxg10','psxg'),('rolling_goals10','is_goal')]:
    shots_s[col] = shots_s.groupby('player_name')[src].transform(
        lambda s: s.rolling(10, min_periods=3).mean())
shots = shots_s.sort_index()
print('Rolling form computed ✓')

## 10 · Save All Data

In [ ]:
shots.to_csv(os.path.join(OUTPUT_DIR,'all_shots_full.csv'), index=False)

player_summary = (
    shots.groupby(['player_name','season'])
    .agg(
        shots_n=('xg','count'), goals=('is_goal','sum'),
        xg=('xg','sum'), psxg=('psxg','sum'), xgot=('xgot','sum'),
        sit_danger=('situation_danger','sum'), on_target=('is_on_target','sum'),
        placement_mean=('placement_score','mean'), power_mean=('shot_power','mean'),
        contact_mean=('contact_quality','mean'), finishing_luck=('finishing_luck','sum'),
    ).reset_index()
)
player_summary['conv_rate']        = player_summary['goals'] / player_summary['shots_n'].clip(1)
player_summary['ot_pct']           = player_summary['on_target'] / player_summary['shots_n'].clip(1)
player_summary['goals_minus_xg']   = player_summary['goals'] - player_summary['xg']
player_summary['goals_minus_psxg'] = player_summary['goals'] - player_summary['psxg']
player_summary.to_csv(os.path.join(OUTPUT_DIR,'player_summary.csv'), index=False)

print(f"Saved all_shots_full.csv ({len(shots):,} rows, {len(shots.columns)} columns)")
print(f"Saved player_summary.csv ({len(player_summary):,} rows)")
print(f"\nNew columns added: {[c for c in shots.columns if c not in ['match_file','season','player_id','player_name','contestant_id','type_id','period_id','time_min','x','y']]}")

## 11 · Model Evaluation Plots

In [ ]:
plt.style.use('dark_background')
fig, axes = plt.subplots(1, 3, figsize=(18, 5), facecolor=FIG_BG)

# ROC curves
ax = axes[0]
for fpr_c, tpr_c, sc, label, color, ls in [
    (xg_fpr,xg_tpr,xg_auc,'Pre-Shot xG',C_BLUE,'-'),
    (psxg_fpr,psxg_tpr,psxg_auc,'Post-Shot psxG',C_GREEN,'-'),
    (pot_fpr,pot_tpr,pot_auc,'P(On Target)',C_PURPLE,'--'),
    (xgot_fpr,xgot_tpr,xgot_auc,'xGOT',C_YELLOW,'-'),
    (sit_fpr,sit_tpr,sit_auc,'Sit. Danger',C_ORANGE,':'),
]:
    ax.plot(fpr_c, tpr_c, lw=2, color=color, ls=ls, label=f'{label} {sc:.3f}')
ax.plot([0,1],[0,1],'w--',lw=1,alpha=0.3)
ax.set(xlabel='FPR',ylabel='TPR',title='ROC — All Models',facecolor=FIG_BG)
ax.legend(fontsize=7)

# Calibration
ax = axes[1]
for proba, label, color, yr in [
    (xg_p,'Pre-Shot xG',C_BLUE,y_test),
    (psxg_p,'Post-Shot psxG',C_GREEN,y_ps.iloc[tp]),
    (pot_p,'P(On Target)',C_PURPLE,y_ot_test),
    (xgot_p,'xGOT',C_YELLOW,y_otg.iloc[to]),
]:
    pt,pp = calibration_curve(yr,proba,n_bins=10,strategy='quantile')
    ax.plot(pp,pt,marker='o',lw=2,color=color,label=label,markersize=3)
ax.plot([0,1],[0,1],'w--',lw=1,alpha=0.3)
ax.set(xlabel='Predicted',ylabel='Actual',title='Calibration',facecolor=FIG_BG)
ax.legend(fontsize=7)

# xG distribution
ax = axes[2]
bins = np.linspace(0,1,25)
ax.hist(y_test[y_test==0].index.map(lambda i: xg_p[np.where(y_test.values==0)[0]]),
        bins=bins, alpha=0.5, color=C_ORANGE, density=True, label='No Goal')
df_eval = pd.DataFrame({'xg':xg_p,'goal':y_test.values})
ax.hist(df_eval[df_eval['goal']==0]['xg'],bins=bins,alpha=0.5,color=C_ORANGE,density=True,label='No Goal')
ax.hist(df_eval[df_eval['goal']==1]['xg'],bins=bins,alpha=0.5,color=C_BLUE,density=True,label='Goal')
ax.set(xlabel='Pre-Shot xG',title='xG Distribution',facecolor=FIG_BG)
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,'model_evaluation.png'),dpi=150,bbox_inches='tight',facecolor=FIG_BG)
plt.show()
print('Model evaluation summary:')
for name,m_auc,m_brier in [
    ('Pre-Shot xG',xg_auc,xg_brier),('Post-Shot psxG',psxg_auc,psxg_brier),
    ('Situation Danger',sit_auc,sit_brier),('P(On Target)',pot_auc,pot_brier),('xGOT',xgot_auc,xgot_brier)
]:
    print(f'  {name:20s}  AUC={m_auc:.4f}  Brier={m_brier:.4f}')

## 12 · Goal Frame Heatmaps

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor=FIG_BG)

def draw_frame(ax):
    ax.add_patch(plt.Rectangle((GOAL_Y_LEFT,0),GOAL_WIDTH,100,
                                fill=False,edgecolor='white',lw=2,zorder=5))
    ax.axvline(GOAL_Y_CENTRE,color='white',ls='--',alpha=0.3,lw=1)
    ax.set_xlim(GOAL_Y_LEFT-1,GOAL_Y_RIGHT+1); ax.set_ylim(-5,105)

for ax, mask, title, cmap in [
    (axes[0], shots['is_goal']==1,   f"Goals ({(shots['is_goal']==1).sum():,})",    'YlOrRd'),
    (axes[1], shots['is_blocked']==1,f"Saves ({(shots['is_blocked']==1).sum():,})", 'Blues'),
]:
    sub = shots[mask & shots['goal_y_raw'].notna() & shots['goal_h_raw'].notna()]
    ax.hist2d(sub['goal_y_raw'], sub['goal_h_raw'], bins=(12,10), cmap=cmap,
              range=[[GOAL_Y_LEFT-1,GOAL_Y_RIGHT+1],[0,100]])
    draw_frame(ax)
    ax.set_title(title,color='white',fontsize=11)
    ax.set_xlabel('Lateral Position'); ax.set_ylabel('Height (0–100)')
    ax.set_facecolor(PITCH_BG)

fig.suptitle('Goal Frame Placement — Goals vs Saves',fontsize=13,color='white',fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,'goal_frame_heatmap.png'),dpi=150,bbox_inches='tight',facecolor=FIG_BG)
plt.show()

## 13 · Multi-Outcome Pitch Maps

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12), facecolor=FIG_BG)
for ax, col, title, cmap in [
    (axes[0,0],'p_goal_mo','P(Goal)',  'YlOrRd'),
    (axes[0,1],'p_save',   'P(Save)',  'Blues'),
    (axes[1,0],'p_miss',   'P(Miss)',  'Oranges'),
    (axes[1,1],'p_post',   'P(Post)',  'Purples'),
]:
    if col not in shots.columns: continue
    pitch = Pitch(pitch_type='opta',pitch_color=PITCH_BG,line_color=LINE_COL,linewidth=0.8)
    pitch.draw(ax=ax)
    bs = pitch.bin_statistic(shots['x'],shots['y'],values=shots[col].fillna(0),
                              statistic='mean',bins=(14,9))
    pitch.heatmap(bs,ax=ax,cmap=cmap)
    ax.set_title(title,fontsize=11,color='white')

fig.suptitle('Multi-Outcome Probability Maps',fontsize=14,fontweight='bold',color='white')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,'multi_outcome_maps.png'),dpi=150,bbox_inches='tight',facecolor=FIG_BG)
plt.show()

## 14 · Shot Archetype Breakdown

In [ ]:
arch_stats = (
    shots.groupby('archetype')
    .agg(n=('is_goal','count'), goals=('is_goal','sum'),
         xg_mean=('xg','mean'), psxg_mean=('psxg','mean'),
         ot_pct=('is_on_target','mean'), power_mean=('shot_power','mean'),
         placement_mean=('placement_score','mean'))
    .reset_index()
)
arch_stats['conv_rate'] = arch_stats['goals'] / arch_stats['n']

fig, axes = plt.subplots(1, 3, figsize=(18, 5), facecolor=FIG_BG)
arch_s = arch_stats.sort_values('conv_rate', ascending=True)
colors = plt.cm.plasma(np.linspace(0.2, 0.9, len(arch_s)))

for ax, col, xlabel in [
    (axes[0],'conv_rate','Conversion Rate'),
    (axes[1],'psxg_mean','Mean psxG'),
    (axes[2],'power_mean','Mean Shot Power'),
]:
    bars = ax.barh(arch_s['archetype'], arch_s[col], color=colors, alpha=0.85)
    ax.set_xlabel(xlabel, fontsize=9); ax.set_facecolor(FIG_BG); ax.tick_params(labelsize=8)
    for bar, (_, row) in zip(bars, arch_s.iterrows()):
        ax.text(bar.get_width()+0.001, bar.get_y()+bar.get_height()/2,
                f"n={row['n']:,}", va='center', fontsize=6.5, color='white')

axes[0].set_title('Shot Archetypes — Conversion Rate', fontsize=10)
axes[1].set_title('Shot Archetypes — Mean psxG', fontsize=10)
axes[2].set_title('Shot Archetypes — Mean Shot Power', fontsize=10)
plt.tight_layout(facecolor=FIG_BG)
plt.savefig(os.path.join(OUTPUT_DIR,'shot_archetypes.png'),dpi=150,bbox_inches='tight',facecolor=FIG_BG)
plt.show()

## 15 · Player Deep-Dive
Change `PLAYER_NAME` in Section 2 to analyse any player.

In [ ]:
player_shots = shots[shots['player_name'].str.contains(PLAYER_NAME, na=False, case=False)].copy()
player_seasons = player_summary[player_summary['player_name'].str.contains(PLAYER_NAME, na=False, case=False)].sort_values('season')

print(f"Player: {PLAYER_NAME}")
print(f"Total shots: {len(player_shots):,}  |  Goals: {player_shots['is_goal'].sum():,}  |  "
      f"Seasons: {player_shots['season'].nunique()}")
print()
display(player_seasons[['season','shots_n','goals','conv_rate','xg','psxg','xgot',
                         'goals_minus_xg','goals_minus_psxg','placement_mean','power_mean']]
        .round(2))

In [ ]:
# Goals vs xG vs psxG vs xGOT bar chart
fig, axes = plt.subplots(1, 2, figsize=(16, 5), facecolor=FIG_BG)

ax = axes[0]
x_pos = np.arange(len(player_seasons))
w = 0.20
ax.bar(x_pos-1.5*w, player_seasons['goals'].values,    w, color=C_YELLOW, alpha=0.9, label='Goals')
ax.bar(x_pos-0.5*w, player_seasons['xg'].values,       w, color=C_BLUE,   alpha=0.8, label='xG')
ax.bar(x_pos+0.5*w, player_seasons['psxg'].values,     w, color=C_GREEN,  alpha=0.8, label='psxG')
ax.bar(x_pos+1.5*w, player_seasons['xgot'].values,     w, color=C_PURPLE, alpha=0.6, label='xGOT')
ax.set_xticks(x_pos)
ax.set_xticklabels(player_seasons['season'].values, rotation=30, ha='right', fontsize=8)
ax.set_title(f'{PLAYER_NAME} — Goals vs xG vs psxG vs xGOT', fontsize=10)
ax.legend(fontsize=8, ncol=4); ax.set_facecolor(FIG_BG)
for i, row in enumerate(player_seasons.itertuples()):
    diff = row.goals - row.psxg
    ax.text(i+0.5*w, max(row.goals,row.psxg)+0.12, f'{diff:+.1f}',
            ha='center', fontsize=7, color=C_GREEN if diff>=0 else C_ORANGE)

# Rolling form
ax2 = axes[1]
ps_sorted = player_shots.sort_values(['season','time_min'])
idx = np.arange(len(ps_sorted))
for col, label, color in [
    ('rolling_xg10','Rolling xG (10)',C_BLUE),
    ('rolling_psxg10','Rolling psxG (10)',C_GREEN),
    ('rolling_goals10','Rolling Goals (10)',C_YELLOW),
]:
    if col in ps_sorted.columns:
        ax2.fill_between(idx, ps_sorted[col].fillna(0).values, alpha=0.3, color=color, label=label)

season_starts = ps_sorted.reset_index(drop=True).groupby('season').apply(lambda df: df.index[0])
for s_idx, s_name in zip(season_starts.values, season_starts.index):
    ax2.axvline(s_idx, color='white', alpha=0.2, lw=0.8)
    ax2.text(s_idx+1, 0.75, s_name.replace('WSL ',''), color='white', fontsize=6, alpha=0.6)
ax2.set(xlabel='Shot number', ylabel='Rolling mean',
        title=f'{PLAYER_NAME} — Career Rolling Shot Quality', facecolor=FIG_BG)
ax2.legend(fontsize=8)

plt.tight_layout(facecolor=FIG_BG)
plt.savefig(os.path.join(OUTPUT_DIR,f'{PLAYER_NAME.lower()}_career.png'),dpi=150,bbox_inches='tight',facecolor=FIG_BG)
plt.show()

In [ ]:
# Shot map + archetype mix
fig, axes = plt.subplots(1, 2, figsize=(16, 8), facecolor=FIG_BG)

# Shot map (sized by psxG, coloured by outcome)
ax = axes[0]
vpitch = VerticalPitch(pitch_type='opta',pitch_color=PITCH_BG,
                       line_color=LINE_COL,linewidth=0.8,half=True)
vpitch.draw(ax=ax)
arch_cmap = plt.cm.tab10(np.linspace(0,1,6))
arch_color_map = {v: arch_cmap[i] for i,(k,v) in enumerate(archetype_names.items())}
for _, row in player_shots.iterrows():
    pv  = max(float(row.get('psxg',0.03)),0.03)
    col = arch_color_map.get(row['archetype'],(0.5,0.5,0.5,1))
    mk  = '*' if row['is_goal'] else 'o'
    ax.scatter(row['y'],row['x'],s=pv*300,color=[col],marker=mk,
               alpha=0.75,linewidths=0.4,edgecolors='white' if row['is_goal'] else 'none',zorder=3)
legend_h = [mpatches.Patch(color=arch_color_map.get(v,'gray'),label=v,alpha=0.8)
            for v in archetype_names.values()]
ax.legend(handles=legend_h,fontsize=6,loc='lower center',facecolor=FIG_BG,labelcolor='white')
ax.set_title(f'{PLAYER_NAME} Shots by Archetype\n(size=psxG, star=goal)',fontsize=10)

# Archetype mix vs league
ax2 = axes[1]
all_arch  = shots[shots['is_penalty']==0]['archetype'].value_counts(normalize=True)
player_arch = player_shots['archetype'].value_counts(normalize=True)
arch_keys = sorted(set(all_arch.index)|set(player_arch.index))
x_a = np.arange(len(arch_keys))
ax2.bar(x_a-0.2,[all_arch.get(k,0) for k in arch_keys],0.4,color=C_MUTED,alpha=0.8,label='All WSL')
ax2.bar(x_a+0.2,[player_arch.get(k,0) for k in arch_keys],0.4,color=C_YELLOW,alpha=0.8,label=PLAYER_NAME)
ax2.set_xticks(x_a)
ax2.set_xticklabels([k.replace(' ','\n') for k in arch_keys],fontsize=7)
ax2.set(ylabel='Share of shots',title='Archetype Mix vs All WSL',facecolor=FIG_BG)
ax2.legend(fontsize=9)

plt.tight_layout(facecolor=FIG_BG)
plt.savefig(os.path.join(OUTPUT_DIR,f'{PLAYER_NAME.lower()}_shotmap.png'),dpi=150,bbox_inches='tight',facecolor=FIG_BG)
plt.show()

In [ ]:
# Shot quality metrics radar
from matplotlib.patches import FancyArrowPatch

# Per-season percentile ranks among all players with >= 20 shots
qualified = player_summary[player_summary['shots_n'] >= 20].copy()
metrics = ['conv_rate','xg','psxg','placement_mean','power_mean','contact_mean','ot_pct']
labels  = ['Conv Rate','xG','psxG','Placement','Shot Power','Contact Quality','On Target %']

for m in metrics:
    qualified[f'{m}_pct'] = qualified[m].rank(pct=True) * 100

player_pct = qualified[
    qualified['player_name'].str.contains(PLAYER_NAME, na=False, case=False)
].sort_values('season')

if len(player_pct) > 0:
    angles = np.linspace(0, 2*np.pi, len(metrics), endpoint=False).tolist()
    angles += angles[:1]

    fig, axes = plt.subplots(2, len(player_pct)//2+1,
                              subplot_kw=dict(polar=True),
                              figsize=(min(22, 4*len(player_pct)), 8),
                              facecolor=FIG_BG)
    axes_flat = np.array(axes).flatten()

    for i, (_, row) in enumerate(player_pct.iterrows()):
        if i >= len(axes_flat): break
        ax = axes_flat[i]
        vals = [row.get(f'{m}_pct', 50) for m in metrics]
        vals += vals[:1]
        ax.plot(angles, vals, color=C_BLUE, lw=2)
        ax.fill(angles, vals, color=C_BLUE, alpha=0.25)
        ax.set_thetagrids(np.degrees(angles[:-1]), labels, size=7, color='white')
        ax.set_ylim(0,100)
        ax.set_title(row['season'].replace('WSL ',''), size=9, color='white', pad=10)
        ax.set_facecolor(FIG_BG)
        ax.grid(color='white', alpha=0.2)
        ax.tick_params(colors='white')
        ax.axhline(50, color='white', alpha=0.2, lw=0.8)

    for j in range(i+1, len(axes_flat)):
        axes_flat[j].set_visible(False)

    fig.suptitle(f'{PLAYER_NAME} — Percentile Rankings vs All WSL (≥20 shots)',
                 fontsize=13, color='white', fontweight='bold')
    plt.tight_layout(facecolor=FIG_BG)
    plt.savefig(os.path.join(OUTPUT_DIR,f'{PLAYER_NAME.lower()}_radar.png'),dpi=150,bbox_inches='tight',facecolor=FIG_BG)
    plt.show()
else:
    print(f'No seasons with ≥20 shots found for {PLAYER_NAME}')

## 16 · SHAP Feature Explanations
> Optional — requires `shap` package (installed in Section 1)

In [ ]:
if HAS_SHAP:
    fig, axes = plt.subplots(1, 3, figsize=(20, 6), facecolor=FIG_BG)
    for ax, model, feat_names, title in [
        (axes[0], xg_model,   XG_FEATURES,   'Pre-Shot xG'),
        (axes[1], psxg_model, PSXG_FEATURES, 'Post-Shot psxG'),
        (axes[2], sit_model,  SIT_FEATURES,  'Situation Danger'),
    ]:
        sample = X_base_test.sample(min(1500,len(X_base_test)),random_state=0) \
                 if 'base' in str(feat_names) else \
                 X_sit_test.sample(min(1500,len(X_sit_test)),random_state=0)
        if feat_names == PSXG_FEATURES:
            sample = X_ps.iloc[tp].sample(min(1500,len(tp)),random_state=0)
        explainer = shap.TreeExplainer(model.clf)
        sv = explainer.shap_values(sample)
        imp = pd.Series(np.abs(sv).mean(0), index=feat_names).sort_values(ascending=True).tail(12)
        ax.barh(imp.index, imp.values, color=C_BLUE, alpha=0.8)
        ax.set_title(f'SHAP — {title}', fontsize=10)
        ax.set_facecolor(FIG_BG); ax.tick_params(labelsize=7)
    plt.tight_layout(facecolor=FIG_BG)
    plt.savefig(os.path.join(OUTPUT_DIR,'shap_summary.png'),dpi=150,bbox_inches='tight',facecolor=FIG_BG)
    plt.show()
else:
    print('Install shap to see feature explanations: !pip install shap')

## 17 · League-Wide Top Finishers

In [ ]:
# Top 20 players by goals - psxG (overperformers) — minimum 30 shots
career = (
    shots.groupby('player_name')
    .agg(shots_n=('xg','count'), goals=('is_goal','sum'),
         xg=('xg','sum'), psxg=('psxg','sum'), xgot=('xgot','sum'),
         placement_mean=('placement_score','mean'),
         power_mean=('shot_power','mean'),
         contact_mean=('contact_quality','mean'))
    .reset_index()
)
career['conv_rate']        = career['goals'] / career['shots_n'].clip(1)
career['goals_minus_psxg'] = career['goals'] - career['psxg']
career['xg_per_shot']      = career['xg']   / career['shots_n'].clip(1)

qualified_c = career[career['shots_n'] >= 30].copy()

fig, axes = plt.subplots(1, 2, figsize=(16, 8), facecolor=FIG_BG)

# Top finishers: goals - psxG
top_fin = qualified_c.sort_values('goals_minus_psxg', ascending=False).head(20)
colors_f = [C_YELLOW if PLAYER_NAME in n else C_BLUE for n in top_fin['player_name']]
bars = axes[0].barh(top_fin['player_name'], top_fin['goals_minus_psxg'], color=colors_f, alpha=0.85)
axes[0].axvline(0, color='white', lw=0.8, alpha=0.4)
axes[0].set(xlabel='Goals − psxG', title='Top 20 Finishers\n(Goals above psxG, min 30 shots)', facecolor=FIG_BG)
axes[0].tick_params(labelsize=8)
for bar, (_, row) in zip(bars, top_fin.iterrows()):
    axes[0].text(bar.get_width()+0.05, bar.get_y()+bar.get_height()/2,
                 f"{row['goals']:.0f}G / {row['psxg']:.1f}psxG",
                 va='center', fontsize=6.5, color='white')

# Best contact quality
top_cq = qualified_c.sort_values('contact_mean', ascending=False).head(20)
colors_c = [C_YELLOW if PLAYER_NAME in n else C_GREEN for n in top_cq['player_name']]
axes[1].barh(top_cq['player_name'], top_cq['contact_mean'], color=colors_c, alpha=0.85)
axes[1].set(xlabel='Mean Contact Quality Index', title='Top 20 Contact Quality\n(min 30 shots)', facecolor=FIG_BG)
axes[1].tick_params(labelsize=8)

plt.tight_layout(facecolor=FIG_BG)
plt.savefig(os.path.join(OUTPUT_DIR,'league_rankings.png'),dpi=150,bbox_inches='tight',facecolor=FIG_BG)
plt.show()

print('\nTop 10 finishers (Goals − psxG):')
display(top_fin[['player_name','shots_n','goals','xg','psxg','goals_minus_psxg','conv_rate']].head(10).round(2))

## 18 · Pressure Efficiency

In [ ]:
peff = (
    shots[shots['is_penalty']==0]
    .groupby(['player_name','under_pressure'])
    .agg(n=('xg','count'), goals=('is_goal','sum'), xg_mean=('xg','mean'), psxg_mean=('psxg','mean'))
    .reset_index()
)
peff['conv_rate'] = peff['goals'] / peff['n'].clip(1)
peff_p = peff.pivot_table(index='player_name', columns='under_pressure',
                          values=['conv_rate','xg_mean','n'], aggfunc='first').reset_index()
peff_p.columns = ['player_name','conv_open','conv_pressure','xg_open','xg_pressure','n_open','n_pressure']
peff_p['pressure_drop'] = peff_p['conv_open'].fillna(0) - peff_p['conv_pressure'].fillna(0)
peff_p = peff_p.dropna(subset=['conv_open','conv_pressure'])
peff_top = peff_p[
    peff_p['n_open'].fillna(0)+peff_p['n_pressure'].fillna(0) >= 50
].sort_values('pressure_drop', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(12, 6), facecolor=FIG_BG)
y_pos = np.arange(len(peff_top))
ax.barh(y_pos,     peff_top['conv_open'].values,     0.4, color=C_BLUE,   alpha=0.8, label='Open')
ax.barh(y_pos+0.4, peff_top['conv_pressure'].values, 0.4, color=C_ORANGE, alpha=0.8, label='Under Pressure')
ax.set_yticks(y_pos+0.2)
ax.set_yticklabels(peff_top['player_name'].values, fontsize=8)
ax.set(xlabel='Conversion Rate', title='Pressure vs Open Conversion Rate\n(Top 15 by drop-off, min 50 shots)', facecolor=FIG_BG)
ax.legend(fontsize=9)
plt.tight_layout(facecolor=FIG_BG)
plt.savefig(os.path.join(OUTPUT_DIR,'pressure_efficiency.png'),dpi=150,bbox_inches='tight',facecolor=FIG_BG)
plt.show()

## 19 · Export Final Files
All outputs are saved to your Google Drive `xg_output/` folder.

In [ ]:
# Final save
shots.to_csv(os.path.join(OUTPUT_DIR,'all_shots_full.csv'), index=False)
player_summary.to_csv(os.path.join(OUTPUT_DIR,'player_summary.csv'), index=False)
career.to_csv(os.path.join(OUTPUT_DIR,'career_summary.csv'), index=False)
peff_p.to_csv(os.path.join(OUTPUT_DIR,'pressure_efficiency.csv'), index=False)

print('Files saved to', OUTPUT_DIR)
print()
files_out = glob.glob(os.path.join(OUTPUT_DIR,'*'))
for f in sorted(files_out):
    size = os.path.getsize(f)/1024
    print(f'  {os.path.basename(f):40s}  {size:8.1f} KB')

print(f'\n✓ Done. {len(shots.columns)} columns in all_shots_full.csv')